In [0]:
from pyspark.sql.functions import *
import os
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.window import Window

Visualizações e análises: 
- Caracterização geral dos dados em mãos - distribuição por distritos, tipos de instalaçoes, high limits atingidos, etc etc
- Agrupar limites atingidos por meses/estaçoes
- agrupar limites atingidos por condiçoes metereológicas
- Agrupar limites atingidos por zonas/distritos

In [0]:
df = spark.read.table("hive_metastore.default.model_table_weather")

In [0]:
# Dictionary mapping each Concelho to Distrito
concelho_to_distrito = {
    "Aveiro": [
        "Águeda",
        "Albergaria-a-Velha",
        "Anadia",
        "Aveiro",
        "Castelo de Paiva",
        "Espinho",
        "Estarreja",
        "Ílhavo",
        "Mealhada",
        "Murtosa",
        "Oliveira do Bairro",
        "Ovar",
        "Santa Maria da Feira",
        "São João da Madeira",
        "Sever do Vouga",
        "Vagos",
        "Vale de Cambra"
    ],
    "Beja": [
        "Aljustrel",
        "Almodôvar",
        "Alvito",
        "Barrancos",
        "Beja",
        "Castro Verde",
        "Cuba",
        "Ferreira do Alentejo",
        "Mértola",
        "Moura",
        "Ourique",
        "Serpa",
        "Vidigueira"
    ],
    "Braga": [
        "Amares",
        "Barcelos",
        "Braga",
        "Cabeceiras de Basto",
        "Celorico de Basto",
        "Esposende",
        "Fafe",
        "Guimarães",
        "Póvoa de Lanhoso",
        "Terras de Bouro",
        "Vieira do Minho",
        "Vila Nova de Famalicão",
        "Vila Verde"
    ],
    "Bragança": [
        "Alfândega da Fé",
        "Bragança",
        "Carrazeda de Ansiães",
        "Freixo de Espada à Cinta",
        "Macedo de Cavaleiros",
        "Miranda do Douro",
        "Mirandela",
        "Mogadouro",
        "Vila Flor",
        "Vinhais"
    ],
    "Castelo Branco": [
        "Belmonte",
        "Castelo Branco",
        "Covilhã",
        "Fundão",
        "Idanha-a-Nova",
        "Oleiros",
        "Penamacor",
        "Proença-a-Nova",
        "Sertã",
        "Vila de Rei"
    ],
    "Coimbra": [
        "Arganil",
        "Cantanhede",
        "Coimbra",
        "Condeixa-a-Nova",
        "Figueira da Foz",
        "Lousã",
        "Miranda do Corvo",
        "Montemor-o-Velho",
        "Oliveira do Hospital",
        "Penacova",
        "Tábua",
        "Vila Nova de Poiares"
    ],
    "Évora": [
        "Alandroal",
        "Arraiolos",
        "Borba",
        "Estremoz",
        "Évora",
        "Montemor-o-Novo",
        "Mourão",
        "Redondo",
        "Reguengos de Monsaraz"
    ],
    "Faro": [
        "Albufeira",
        "Alcoutim",
        "Aljezur",
        "Castro Marim",
        "Faro",
        "Lagoa",
        "Lagos",
        "Loulé",
        "Monchique",
        "Olhão",
        "Portimão",
        "São Brás de Alportel",
        "Silves",
        "Tavira",
        "Vila do Bispo"
    ],
    "Guarda": [
        "Almeida",
        "Celorico da Beira",
        "Figueira de Castelo Rodrigo",
        "Fornos de Algodres",
        "Gouveia",
        "Guarda",
        "Manteigas",
        "Mêda",
        "Pinhel",
        "Sabugal",
        "Seia",
        "Trancoso"
    ],
    "Leiria": [
        "Alcobaça",
        "Alvaiázere",
        "Caldas da Rainha",
        "Leiria",
        "Marinha Grande",
        "Nazaré",
        "Óbidos",
        "Pedrógão Grande",
        "Peniche",
        "Pombal",
        "Porto de Mós"
    ],
    "Lisboa": [
        "Alenquer",
        "Amadora",
        "Arruda dos Vinhos",
        "Azambuja",
        "Cadaval",
        "Cascais",
        "Lisboa",
        "Loures",
        "Mafra",
        "Odivelas",
        "Oeiras",
        "Sintra",
        "Torres Vedras",
        "Vila Franca de Xira"
    ],
    "Portalegre": [
        "Arronches",
        "Avis",
        "Castelo de Vide",
        "Crato",
        "Elvas",
        "Fronteira",
        "Marvão",
        "Monforte",
        "Nisa",
        "Portalegre"
    ],
    "Porto": [
        "Amarante",
        "Baião",
        "Felgueiras",
        "Gondomar",
        "Lousada",
        "Maia",
        "Marco de Canaveses",
        "Matosinhos",
        "Paços de Ferreira",
        "Paredes",
        "Penafiel",
        "Porto",
        "Santo Tirso",
        "Trofa",
        "Valongo",
        "Vila do Conde",
        "Vila Nova de Gaia"
    ],
    "Santarém": [
        "Abrantes",
        "Alcanena",
        "Almeirim",
        "Alpiarça",
        "Benavente",
        "Cartaxo",
        "Chamusca",
        "Coruche",
        "Entroncamento",
        "Ferreira do Zêzere",
        "Golegã",
        "Mação",
        "Santarém",
        "Tomar",
        "Torres Novas",
        "Vila Nova da Barquinha"
    ],
    "Setúbal": [
        "Alcácer do Sal",
        "Alcochete",
        "Almada",
        "Barreiro",
        "Grândola",
        "Moita",
        "Montijo",
        "Palmela",
        "Santiago do Cacém",
        "Seixal",
        "Sesimbra",
        "Setúbal",
        "Sines"
    ],
    "Viana do Castelo": [
        "Arcos de Valdevez",
        "Caminha",
        "Melgaço",
        "Monção",
        "Paredes de Coura",
        "Ponte da Barca",
        "Ponte de Lima",
        "Valença",
        "Viana do Castelo",
        "Vila Nova de Cerveira"
    ],
    "Vila Real": [
        "Alijó",
        "Boticas",
        "Chaves",
        "Mesão Frio",
        "Mondim de Basto",
        "Montalegre",
        "Murça",
        "Peso da Régua",
        "Ribeira de Pena",
        "Vila Pouca de Aguiar"
    ],
    "Viseu": [
        "Aguiar da Beira",
        "Armamar",
        "Carregal do Sal",
        "Castro Daire",
        "Cinfães",
        "Lamego",
        "Mangualde",
        "Moimenta da Beira",
        "Mortágua",
        "Nelas",
        "Oliveira de Frades",
        "Penalva do Castelo",
        "Penedono",
        "Resende",
        "Santa Comba Dão",
        "São João da Pesqueira",
        "Sátão",
        "Sernancelhe",
        "Tondela",
        "Viseu",
        "Vouzela"
    ]
}

reverse_concelho_to_distrito = {concelho: distrito for distrito, concelhos in concelho_to_distrito.items() for concelho in concelhos}

# Define a UDF to map Concelho to Distrito
def get_distrito(concelho):
    return reverse_concelho_to_distrito.get(concelho, None)

# Register the UDF
get_distrito_udf = udf(get_distrito, StringType())

# Apply the UDF
df = df.withColumn('DISTRITO', get_distrito_udf(df['Concelho']))

display(df)

In [0]:
# Clean TIPOINST column
df = df.withColumn("TIPOINST", upper(trim(col("TIPOINST"))))

In [0]:
display(df)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
display(df.filter(col("ID").like('%NSPL%')))

In [0]:
df = df.withColumn(
    "OVER_LIMIT_FLAG",
    when((col("TIME_OVER_LIMIT_I") > 0) | (col("TIME_OVER_LIMIT_T") > 0), 1).otherwise(0)
)



In [0]:
display(df)

Databricks visualization. Run in Databricks to view.

In [0]:
display(df.filter(col("over_limit_flag") == 1))

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
ids_to_remove = ["FSALGW-URT-URFCM"]

df = df.filter(~df["TAGCOM"].isin(ids_to_remove))

In [0]:
daily_avg_tension = df.withColumn("DATE_ONLY", to_date("DATE")) \
                      .groupBy("DATE_ONLY") \
                      .agg(
                          avg("TENSION").alias("avg_tension"),
                          avg("INTENSITY").alias("avg_intensity"),
                          avg("temperature").alias("avg_temperature"),
                          avg("humidity").alias("avg_humidity"),
                          avg("precipitation").alias("avg_precipitation"),
                          avg("wind_speed").alias("avg_wind_speed"),
                          avg("TIME_OVER_LIMIT_I").alias("TIME_OVER_LIMIT_I"),
                          avg("TIME_OVER_LIMIT_T").alias("TIME_OVER_LIMIT_T")
                          )

display(daily_avg_tension.orderBy("DATE_ONLY"))

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
display(df.filter(col("INTENSITY") > 1000))

In [0]:
df.select(
    min("DATE").alias("earliest_timestamp"),
    max("DATE").alias("latest_timestamp")
).display()